# CNN-LSTM — Brain Tumour MRI Classification

**Owner:** Member 3 — Pranay Dutta (230953484)  
**Project:** A Comparative Study of Deep Learning Architectures for Brain Tumour MRI Classification (ICT-4442)

Recurrent hybrid: the image is cut into 32×32 patches, a small CNN embeds each patch, an LSTM reads the patch sequence.

All four models share the same data split, preprocessing and training/evaluation code (`common/`); only the architecture and learning rate differ.

> The printed outputs below are from the original run on a Kaggle GPU. The last cell writes the files the comparison scripts in `analysis/` read.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root, so `common` and `models` can be imported

import torch
from common.config import CLASSES, OUTPUT_DIR, device, set_seed
from common.data import describe_dataset, get_loaders
from common.engine import train_model, evaluate_model, get_probs_and_labels
from common.results import save_run

print(f"Device: {device}")
set_seed()

Device: cuda


## 2. Data (shared split)

In [2]:
describe_dataset()

Train classes found: ['glioma', 'meningioma', 'notumor', 'pituitary']
Test classes found:  ['glioma', 'meningioma', 'notumor', 'pituitary']
  glioma: 1400 images
  meningioma: 1400 images
  notumor: 1400 images
  pituitary: 1400 images


In [3]:
train_loader, val_loader, test_loader = get_loaders()

Train: 5040 | Val: 560 | Test: 1600


## 3. Model — CNN-LSTM

In [4]:
from models.cnn_lstm import CNNLSTM

# Sanity check the patching logic on one dummy batch before training
dummy = torch.randn(2, 3, 224, 224)
test_model = CNNLSTM()
out = test_model(dummy)
print(f"CNN-LSTM output shape: {out.shape}  (expected: [2, 4])")

n_params = sum(p.numel() for p in CNNLSTM().parameters())
print(f"CNN-LSTM parameters: {n_params:,}")

CNN-LSTM output shape: torch.Size([2, 4])  (expected: [2, 4])
CNN-LSTM parameters: 88,548


## 4. Train (10 epochs, Adam, cross-entropy) and evaluate on the test set

In [5]:
cnnlstm_model = CNNLSTM(num_classes=len(CLASSES))
cnnlstm_model, cnnlstm_history = train_model(cnnlstm_model, train_loader, val_loader, epochs=10, model_name="CNN-LSTM")

cnnlstm_loss, cnnlstm_acc, cnnlstm_prec, cnnlstm_rec, cnnlstm_f1, cnnlstm_cm = evaluate_model(cnnlstm_model, test_loader)
print(f"\nCNN-LSTM Test Results — Acc: {cnnlstm_acc:.4f}  Precision: {cnnlstm_prec:.4f}  Recall: {cnnlstm_rec:.4f}  F1: {cnnlstm_f1:.4f}")

torch.save(cnnlstm_model.state_dict(), OUTPUT_DIR / 'cnnlstm_final.pth')

[CNN-LSTM] Epoch 1/10  Train Loss: 1.3306  Train Acc: 0.2851  Val Loss: 1.1847  Val Acc: 0.4268
[CNN-LSTM] Epoch 2/10  Train Loss: 1.1386  Train Acc: 0.4476  Val Loss: 1.1197  Val Acc: 0.4821
[CNN-LSTM] Epoch 3/10  Train Loss: 1.0811  Train Acc: 0.5135  Val Loss: 1.0979  Val Acc: 0.5089
[CNN-LSTM] Epoch 4/10  Train Loss: 1.0418  Train Acc: 0.5300  Val Loss: 1.0272  Val Acc: 0.5482
[CNN-LSTM] Epoch 5/10  Train Loss: 1.0155  Train Acc: 0.5542  Val Loss: 1.0018  Val Acc: 0.5821
[CNN-LSTM] Epoch 6/10  Train Loss: 0.9920  Train Acc: 0.5833  Val Loss: 0.9676  Val Acc: 0.6071
[CNN-LSTM] Epoch 7/10  Train Loss: 0.9167  Train Acc: 0.6323  Val Loss: 0.8701  Val Acc: 0.6589
[CNN-LSTM] Epoch 8/10  Train Loss: 0.8605  Train Acc: 0.6490  Val Loss: 0.9122  Val Acc: 0.6268
[CNN-LSTM] Epoch 9/10  Train Loss: 0.8177  Train Acc: 0.6730  Val Loss: 0.8063  Val Acc: 0.6661
[CNN-LSTM] Epoch 10/10  Train Loss: 0.7915  Train Acc: 0.6774  Val Loss: 0.8049  Val Acc: 0.6768

CNN-LSTM Test Results — Acc: 0.6088  P

## 5. Save results for the comparison scripts

In [6]:
save_run("cnn_lstm", cnnlstm_model, cnnlstm_history, test_loader)